# Module 9: Capstone Project
# Building a RAG-Based Dialogue System

## Project Overview

In this capstone project, you will build a complete **Retrieval-Augmented Generation (RAG)** dialogue system that can:

1. **Ingest documents** from various sources
2. **Create vector embeddings** for semantic search
3. **Retrieve relevant context** based on user questions
4. **Generate grounded responses** using an LLM
5. **Maintain conversation history** for multi-turn dialogue

This project integrates everything you've learned throughout the course!

---

## Learning Objectives

By completing this project, you will demonstrate:
- Understanding of **NLP fundamentals** (tokenization, embeddings)
- Proficiency with **LLM APIs** (OpenAI or alternatives)
- **Prompt engineering** skills for conversational AI
- **RAG architecture** implementation
- End-to-end **system integration**

---

## Part 1: Environment Setup

In [ ]:
# Install required packages
!pip install -q openai langchain langchain-openai langchain-community chromadb pypdf python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# For Colab, uncomment and set your API key:
# os.environ['OPENAI_API_KEY'] = 'your-api-key-here'

print("✅ Environment configured!")

In [ ]:
# Imports
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain.schema import Document

print("✅ All imports successful!")

---

## Part 2: Document Ingestion

For this project, we'll create a sample knowledge base. In a real scenario, you would load from PDFs, databases, or APIs.

### Task 2.1: Create Your Knowledge Base

You can either:
- Use the sample documents below
- Load your own PDF files
- Create custom domain-specific content

In [ ]:
# Sample knowledge base: AI Company FAQ
sample_documents = [
    {
        "content": """
        ## About TechAI Corporation
        TechAI Corporation was founded in 2020 and is headquartered in San Francisco, California.
        We specialize in developing enterprise AI solutions including natural language processing,
        computer vision, and predictive analytics platforms. Our mission is to democratize AI
        and make it accessible to businesses of all sizes.
        
        Our flagship product, AIAssist Pro, helps companies automate customer support,
        document processing, and data analysis tasks. We currently serve over 500 enterprise
        clients across 30 countries.
        """,
        "source": "company_overview.md"
    },
    {
        "content": """
        ## Pricing and Plans
        
        ### Starter Plan - $99/month
        - Up to 1,000 API calls per month
        - Email support
        - Access to basic models
        - 1 user seat
        
        ### Professional Plan - $499/month
        - Up to 10,000 API calls per month
        - Priority email and chat support
        - Access to all models including GPT-4
        - 5 user seats
        - Custom model fine-tuning
        
        ### Enterprise Plan - Custom pricing
        - Unlimited API calls
        - 24/7 dedicated support
        - On-premise deployment option
        - Unlimited user seats
        - Custom integrations and SLA
        
        All plans include a 14-day free trial. Annual billing receives 20% discount.
        """,
        "source": "pricing.md"
    },
    {
        "content": """
        ## Technical Documentation
        
        ### API Authentication
        All API requests require authentication using Bearer tokens.
        Generate your API key from the dashboard at dashboard.techai.com.
        
        Include the token in the Authorization header:
        ```
        Authorization: Bearer YOUR_API_KEY
        ```
        
        ### Rate Limits
        - Starter: 10 requests/minute
        - Professional: 60 requests/minute
        - Enterprise: Custom limits
        
        ### Endpoints
        - POST /v1/completions - Text generation
        - POST /v1/embeddings - Create embeddings
        - POST /v1/chat - Conversational AI
        - GET /v1/models - List available models
        
        ### Error Handling
        - 401: Invalid API key
        - 429: Rate limit exceeded
        - 500: Server error
        """,
        "source": "api_documentation.md"
    },
    {
        "content": """
        ## Frequently Asked Questions
        
        **Q: How do I reset my password?**
        A: Go to login.techai.com and click "Forgot Password". Enter your email
        and follow the instructions sent to your inbox.
        
        **Q: Can I upgrade my plan mid-month?**
        A: Yes! You can upgrade anytime. We'll pro-rate the difference for the
        remaining days in your billing cycle.
        
        **Q: What programming languages do you support?**
        A: We provide official SDKs for Python, JavaScript, Go, and Java.
        Our REST API can be used with any language that supports HTTP requests.
        
        **Q: Is my data secure?**
        A: Yes. We are SOC 2 Type II certified and GDPR compliant. All data is
        encrypted at rest and in transit. We do not use customer data for training.
        
        **Q: How do I contact support?**
        A: Email support@techai.com or use the chat widget on our website.
        Enterprise customers have access to dedicated support channels.
        """,
        "source": "faq.md"
    },
    {
        "content": """
        ## Data Security and Compliance
        
        TechAI Corporation takes data security seriously. Our infrastructure
        is hosted on AWS with multi-region redundancy.
        
        ### Certifications
        - SOC 2 Type II
        - GDPR Compliant
        - HIPAA Compliant (Enterprise plan)
        - ISO 27001
        
        ### Data Retention
        - API logs: 30 days
        - Conversation history: Configurable (default 90 days)
        - Fine-tuning data: Deleted after training completes
        
        ### Data Processing
        We process data in the following regions:
        - US (default)
        - EU (available on request)
        - Asia-Pacific (Enterprise only)
        
        For data deletion requests, email privacy@techai.com.
        """,
        "source": "security.md"
    }
]

# Convert to LangChain documents
documents = [
    Document(page_content=doc["content"].strip(), metadata={"source": doc["source"]})
    for doc in sample_documents
]

print(f"✅ Loaded {len(documents)} documents")
for doc in documents:
    print(f"   - {doc.metadata['source']}: {len(doc.page_content)} characters")

### Task 2.2: Document Chunking

Split documents into smaller chunks for better retrieval.

In [ ]:
# Configure text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # Maximum characters per chunk
    chunk_overlap=50,     # Overlap between chunks for context preservation
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

# Split documents
chunks = text_splitter.split_documents(documents)

print(f"✅ Split into {len(chunks)} chunks")
print(f"\nSample chunk:")
print(f"   Source: {chunks[0].metadata['source']}")
print(f"   Content: {chunks[0].page_content[:200]}...")

---

## Part 3: Vector Store and Embeddings

Create embeddings and store them in a vector database for semantic search.

In [ ]:
# Initialize embeddings model
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Create vector store
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="capstone_kb",
    persist_directory="./capstone_vectordb"
)

print(f"✅ Vector store created with {vectorstore._collection.count()} vectors")

In [ ]:
# Test retrieval
test_query = "What is the cost of the professional plan?"

results = vectorstore.similarity_search(test_query, k=3)

print(f"Query: '{test_query}'\n")
print("Retrieved chunks:")
for i, doc in enumerate(results, 1):
    print(f"\n{i}. [{doc.metadata['source']}]")
    print(f"   {doc.page_content[:200]}...")

---

## Part 4: Conversational Chain

Build a conversational chain that maintains history and uses retrieval.

In [ ]:
# Initialize LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7
)

# Create memory for conversation history
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)

# Create conversational retrieval chain
conversation_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    memory=memory,
    return_source_documents=True,
    verbose=False
)

print("✅ Conversational chain ready!")

In [ ]:
# Helper function for chat
def chat(question):
    """Send a message and get a response"""
    result = conversation_chain.invoke({"question": question})
    
    print(f"👤 User: {question}")
    print(f"\n🤖 Assistant: {result['answer']}")
    
    # Show sources
    if result.get('source_documents'):
        sources = set(doc.metadata['source'] for doc in result['source_documents'])
        print(f"\n📚 Sources: {', '.join(sources)}")
    
    print("\n" + "="*60)

---

## Part 5: Test Your Dialogue System

Now let's test the system with a multi-turn conversation!

In [ ]:
# Conversation 1: About the company
chat("Tell me about TechAI Corporation. When were they founded?")

In [ ]:
# Conversation 2: Follow-up question (tests memory)
chat("What is their main product?")

In [ ]:
# Conversation 3: Pricing question
chat("I'm interested in the Professional plan. What features does it include?")

In [ ]:
# Conversation 4: Technical question
chat("How do I authenticate API requests?")

In [ ]:
# Conversation 5: Follow-up with context
chat("What happens if I exceed the rate limits for my plan?")

In [ ]:
# Conversation 6: Security question
chat("Is TechAI HIPAA compliant? What certifications do they have?")

---

## Part 6: Interactive Chat Interface

Create a simple interactive loop for testing.

In [ ]:
def interactive_chat():
    """Interactive chat loop"""
    print("="*60)
    print("🤖 TechAI Support Assistant")
    print("   Type 'quit' to exit, 'reset' to clear history")
    print("="*60)
    print()
    
    while True:
        user_input = input("👤 You: ").strip()
        
        if user_input.lower() == 'quit':
            print("\nGoodbye! 👋")
            break
        elif user_input.lower() == 'reset':
            memory.clear()
            print("\n🔄 Conversation history cleared.\n")
            continue
        elif not user_input:
            continue
        
        result = conversation_chain.invoke({"question": user_input})
        print(f"\n🤖 Assistant: {result['answer']}")
        
        sources = set(doc.metadata['source'] for doc in result.get('source_documents', []))
        if sources:
            print(f"   📚 Sources: {', '.join(sources)}")
        print()

# Uncomment to run interactive chat
# interactive_chat()

---

## Part 7: Extensions and Improvements

### 🚀 Challenge Tasks

Take your dialogue system to the next level with these extensions:

#### Challenge 1: Add Source Citation
Modify the response to include inline citations [1], [2] that reference source documents.

In [ ]:
# Challenge 1: Implement source citations
# TODO: Create a custom prompt template that instructs the model to cite sources

citation_prompt = """
Answer the question based on the context provided.
When using information from the context, cite the source using [Source: filename].

Context:
{context}

Question: {question}

Answer with citations:
"""

# Your implementation here

#### Challenge 2: Add Guardrails
Implement input/output guardrails to handle off-topic questions gracefully.

In [ ]:
# Challenge 2: Implement guardrails
# TODO: Add a check for off-topic questions

def is_on_topic(question, retrieved_docs):
    """Check if question is related to the knowledge base"""
    # Hint: Use similarity scores or ask the LLM to classify
    pass

# Your implementation here

#### Challenge 3: Add Feedback Collection
Implement a simple feedback mechanism to collect user satisfaction ratings.

In [ ]:
# Challenge 3: Feedback collection
# TODO: Store feedback for each response

feedback_log = []

def log_feedback(question, answer, rating, comment=""):
    """Log user feedback for a response"""
    feedback_log.append({
        "question": question,
        "answer": answer[:100] + "...",
        "rating": rating,  # 1-5
        "comment": comment
    })

# Your implementation here

#### Challenge 4: Load Your Own Documents
Replace the sample knowledge base with your own documents.

In [ ]:
# Challenge 4: Load custom documents
# TODO: Load PDFs, markdown files, or other documents

# Example: Load a PDF
# from langchain_community.document_loaders import PyPDFLoader
#
# loader = PyPDFLoader("your_document.pdf")
# custom_docs = loader.load()
#
# print(f"Loaded {len(custom_docs)} pages from PDF")

# Your implementation here

---

## Part 8: Evaluation Criteria

Your capstone project will be evaluated on:

### Core Requirements ✅

| Criterion | Points | Description |
|-----------|--------|-------------|
| Document Ingestion | 15 | Successfully load and chunk documents |
| Vector Store | 15 | Create embeddings and store in vector DB |
| Retrieval | 20 | Retrieve relevant documents for queries |
| Generation | 20 | Generate accurate, grounded responses |
| Conversation History | 15 | Maintain context across turns |
| Code Quality | 15 | Clean, documented, working code |

### Bonus Points 🌟

| Extension | Points |
|-----------|--------|
| Source citations | +10 |
| Guardrails | +10 |
| Custom knowledge base | +10 |
| Feedback system | +5 |
| Error handling | +5 |

---

## 🎯 Key Takeaways

Congratulations on completing the capstone project! You've built a production-ready RAG system that demonstrates:

1. **Document Processing**: Chunking and preparing text for ML
2. **Embedding Generation**: Converting text to vector representations
3. **Semantic Search**: Finding relevant content based on meaning
4. **LLM Integration**: Using chat models for generation
5. **Memory Management**: Maintaining conversation context
6. **System Integration**: Combining multiple components

### What's Next?

To take this further, consider:
- Deploying as a web API with FastAPI
- Adding a Gradio or Streamlit UI
- Implementing evaluation metrics (RAGAS)
- Fine-tuning the retrieval or generation models
- Adding multi-modal support (images, tables)

---

### 🎉 Course Complete!

You've now completed the Hands-On Generative AI course. You have the skills to:
- Build ML and deep learning models
- Work with LLMs and prompt engineering
- Implement RAG systems
- Deploy and evaluate GenAI applications

Keep building and experimenting! 🚀